# KDB-X Financial Analytics Demo

This notebook demonstrates KDB-X's financial analytics capabilities:
- **Time-series analytics** with `xbar` (time bucketing) and `mavg` (moving averages)
- **As-of joins** (`aj`) for point-in-time lookups
- **Vectorised backtesting** with transaction costs
- **Co-temporal enrichment** of training data with market context

**Prerequisites:** KDB-X running via `docker-compose`, market data loaded.

In [ ]:
# Cell 1: Setup & Connect
import os
import sys

# Add project root to path
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from kdbx.connection import pykx_connection
from kdbx.market_tables import create_market_tables, load_parquet_data
from kdbx.backtest import run_backtest
from kdbx.enrichment import enrich_training_pair

# Verify connection
with pykx_connection() as q:
    tables = q("tables[]")
    print("Connected to KDB-X")
    print(f"Tables: {[str(t) for t in tables]}")

In [ ]:
# Cell 2: Load Market Data
# Create tables and load Parquet data
create_market_tables(drop_existing=True)
counts = load_parquet_data("../data/sample_market_data.parquet")

print(f"Loaded rows: {counts}")

# Sample data
with pykx_connection() as q:
    sample = q("select [5] from market_ticks").pd()
    print(f"\nmarket_ticks sample:")
    display(sample)

    book_sample = q("select [5] from order_book").pd()
    print(f"\norder_book sample:")
    display(book_sample)

In [ ]:
# Cell 3: Time-Series Analytics — xbar + mavg
with pykx_connection() as q:
    # 5-minute OHLCV bars using xbar
    bars_5m = q(
        "select open:first open, high:max high, low:min low, close:last close, "
        "vol:sum volume "
        "by sym, 5 xbar timestamp.minute "
        "from market_ticks where sym = `AAPL, "
        "timestamp within (2025.06.16D09:30:00.000000000; 2025.06.16D16:00:00.000000000)"
    ).pd()

    # 20-day moving average — two-step: aggregate daily, then apply mavg
    # mavg must run ACROSS daily rows, not inside a by-group (where it would
    # see only a single scalar per date and return that scalar unchanged).
    aapl_daily = q(
        "update ma20: mavg[20; close] from "
        "select close:last close by date:timestamp.date "
        "from market_ticks where sym = `AAPL"
    ).pd()

# Plot AAPL close + 20-day MA
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(aapl_daily.index, aapl_daily["close"], label="AAPL Close", alpha=0.7)
ax.plot(aapl_daily.index, aapl_daily["ma20"], label="20-day MA", linewidth=2)
ax.set_title("AAPL Daily Close with 20-Day Moving Average")
ax.set_xlabel("Date")
ax.set_ylabel("Price ($)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"5-minute bars sample (AAPL Jun 16):")
display(bars_5m.head(10))

In [ ]:
# Cell 4: Generate Trading Signals
# Raw q — demonstrates q expressiveness for financial data generation
with pykx_connection() as q:
    q(
        "signals: ([] "
        "signal_id: `$string each til 500; "
        "timestamp: 500?exec timestamp from market_ticks; "
        "sym: 500?`AAPL`NVDA`MSFT`GOOG`AMZN; "
        "direction: 500?`BUY`SELL; "
        "confidence: 500?1.0; "
        "model_id: 500#enlist `demo_model; "
        "rationale: 500#enlist \"momentum signal\"; "
        "realized_pnl: 500#0n; "
        "realized_at: 500#0Np)"
    )

    sig_count = int(q("count signals").py())
    sig_sample = q("select [10] from signals").pd()

print(f"Generated {sig_count} trading signals")
display(sig_sample)

In [ ]:
# Cell 5: As-Of Join — Match Signals to Market State
with pykx_connection() as q:
    # aj: for each signal, find the most recent market tick and order book state
    enriched = q(
        "aj[`sym`timestamp; "
        "select signal_id, sym, timestamp, direction, confidence from signals; "
        "select sym, timestamp, close, volume from market_ticks]"
    ).pd()

    book_enriched = q(
        "aj[`sym`timestamp; "
        "select signal_id, sym, timestamp, direction from signals; "
        "select sym, timestamp, bid_price, ask_price, spread from order_book]"
    ).pd()

print(f"Enriched {len(enriched)} signals with market context")
display(enriched.head())

# Scatter: confidence vs volume by direction
fig, ax = plt.subplots(figsize=(10, 6))
for direction, color in [("BUY", "green"), ("SELL", "red")]:
    mask = enriched["direction"] == direction
    subset = enriched[mask]
    ax.scatter(subset["confidence"], subset["volume"],
              alpha=0.3, c=color, label=direction, s=20)

ax.set_xlabel("Signal Confidence")
ax.set_ylabel("Volume at Signal Time")
ax.set_title("Signal Confidence vs. Market Volume (via aj[])")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 6: Vectorised Backtest
results = run_backtest(model_id="demo_model", cost_bps=5.0)

print("Backtest Results (demo_model, 5 bps cost):")
print(f"  Sharpe Ratio:   {results['sharpe']:.4f}")
print(f"  Max Drawdown:   {results['max_drawdown']:.4f}")
print(f"  Total Return:   {results['total_return']:.4f}")
print(f"  Win Rate:       {results['win_rate']:.4f}")
print(f"  Num Trades:     {results['n_trades']}")

# Run with higher cost
results_high_cost = run_backtest(model_id="demo_model", cost_bps=20.0)
print(f"\nWith 20 bps cost:")
print(f"  Sharpe Ratio:   {results_high_cost['sharpe']:.4f}")
print(f"  Total Return:   {results_high_cost['total_return']:.4f}")

In [ ]:
# Cell 7: Co-Temporal Enrichment
# Sample training record — before enrichment
training_record = {
    "question": "What is AAPL's outlook?",
    "answer": "Apple shows strong momentum.",
    "model_score": 0.85,
}

print("BEFORE enrichment:")
for k, v in training_record.items():
    print(f"  {k}: {v}")

# Enrich with point-in-time market context
enriched_record = enrich_training_pair(
    record=training_record,
    sym="AAPL",
    event_timestamp="2025-06-16T10:30:00",
)

print("\nAFTER enrichment:")
for k, v in enriched_record.items():
    print(f"  {k}: {v}")

print(f"\nAdded {len(enriched_record) - len(training_record)} market context fields")

In [ ]:
# Cell 8: Before vs After — Baseline vs Enriched Metrics
# Compare backtest results at different cost levels
cost_levels = [1.0, 5.0, 10.0, 20.0]
sharpes = []
returns = []

for cost in cost_levels:
    r = run_backtest(model_id="demo_model", cost_bps=cost)
    sharpes.append(r["sharpe"])
    returns.append(r["total_return"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sharpe by cost
x = np.arange(len(cost_levels))
width = 0.5
axes[0].bar(x, sharpes, width, color=["#2ecc71", "#3498db", "#f39c12", "#e74c3c"])
axes[0].set_xticks(x)
axes[0].set_xticklabels([f"{c} bps" for c in cost_levels])
axes[0].set_ylabel("Sharpe Ratio")
axes[0].set_title("Sharpe Ratio vs Transaction Cost")
axes[0].grid(True, alpha=0.3, axis="y")

# Total return by cost
axes[1].bar(x, [r * 100 for r in returns], width,
           color=["#2ecc71", "#3498db", "#f39c12", "#e74c3c"])
axes[1].set_xticks(x)
axes[1].set_xticklabels([f"{c} bps" for c in cost_levels])
axes[1].set_ylabel("Total Return (%)")
axes[1].set_title("Total Return vs Transaction Cost")
axes[1].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

print("\nSummary: KDB-X financial analytics capabilities demonstrated:")
print("  1. xbar — time bucketing for OHLCV aggregation")
print("  2. mavg — moving averages for trend analysis")
print("  3. aj[] — as-of joins for point-in-time lookups")
print("  4. Vectorised backtesting with parameterized q")
print("  5. Co-temporal enrichment of training data")